Load Era5 hourly data, pressure data for calculating wind shear

In [1]:
import sys
import xarray as xr
import matplotlib.pyplot as plt

sys.path.append("..")

from src.data.load_era5 import load_era5, convert_longitude, subset_area

# Load all three datasets
ds = load_era5("../data/raw/era5_sample")
ds_pressure = load_era5("../data/raw/era5_pressure_sample.nc")
ds_cape = load_era5("../data/raw/era5_cape_sample.nc")

# Clean up wind pressure 
ds_pressure = convert_longitude(ds_pressure)
ds_pressure = subset_area(ds_pressure, north=60, south=48, west=-10, east=2)

# Clean up cape
ds_cape = convert_longitude(ds_cape)
ds_cape = subset_area(ds_cape, north=60, south=48, west=-10, east=2)

# Merge everything into one dataset
ds_merged = xr.merge([ds, ds_pressure, ds_cape]).sortby("pressure_level", ascending=False)
print(ds_merged)

/home/alf_walks/projects/python/HPCWeatherPredictionPipeline/notebooks/../src/data/load_era5.py:9: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds = xr.open_mfdataset(files, combine="by_coords")
/home/alf_walks/projects/python/HPCWeatherPredictionPipeline/notebooks/../src/data/load_era5.py:9: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitl

<xarray.Dataset> Size: 4MB
Dimensions:         (valid_time: 12, latitude: 49, longitude: 49,
                     pressure_level: 8)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 96B 2020-01-01 ... 2020-01-03...
    expver          (valid_time) <U4 192B '0001' '0001' '0001' ... '0001' '0001'
  * latitude        (latitude) float64 392B 60.0 59.75 59.5 ... 48.5 48.25 48.0
  * longitude       (longitude) float64 392B -10.0 -9.75 -9.5 ... 1.5 1.75 2.0
  * pressure_level  (pressure_level) float64 64B 1e+03 925.0 ... 400.0 300.0
    number          int64 8B 0
Data variables:
    u10             (valid_time, latitude, longitude) float32 115kB dask.array<chunksize=(12, 49, 49), meta=np.ndarray>
    v10             (valid_time, latitude, longitude) float32 115kB dask.array<chunksize=(12, 49, 49), meta=np.ndarray>
    d2m             (valid_time, latitude, longitude) float32 115kB dask.array<chunksize=(12, 49, 49), meta=np.ndarray>
    t2m             (valid_time, latitude, longit

/tmp/ipykernel_4421/189555802.py:23: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds_merged = xr.merge([ds, ds_pressure, ds_cape]).sortby("pressure_level", ascending=False)
/tmp/ipykernel_4421/189555802.py:23: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds_merged = xr.merge([ds, ds_pressure, ds_cape]).sortby("pressure_level", ascen

Calcuate bulk wind shear

In [2]:
import sys
import xarray as xr
import matplotlib.pyplot as plt
sys.path.append("..")

from src.data.load_era5 import load_era5, convert_longitude, subset_area
from src.features.meteorology import compute_shear_features

# compute all shear features
shear_features = compute_shear_features(ds_merged)

#TEST FIGURES

#fig1, ax1 = plt.subplots()
#shear_features['bulk_shear_850_500'].isel(valid_time=7).plot(ax=ax1)
#ax1.set_title("Bulk Shear 850hPa - 500hPa")

#fig2, ax2 = plt.subplots()
#shear_features['dir_shear_850_500'].isel(valid_time=7).plot(ax=ax2, cmap="RdBu", vmin=-180, vmax=180)
#ax2.set_title("Directional Shear 850hPa - 500hPa")

#plt.show()

Calculate Convective Available Potential Energy (CAPE)

In [ ]:
#OPTIONAL CELL
#Use for showing improvement on time taken with dask processes from sequential

import time
import numpy as np
import dask
from dask import delayed
from src.features.meteorology import (
    compute_cape_sequential,
    compute_cape_for_timestep,
)

n_time = len(ds_merged['valid_time'])

start = time.time()
result_seq = compute_cape_sequential(ds_merged)
time_seq = time.time() - start

start = time.time()
results_processes = dask.compute(
    *[delayed(compute_cape_for_timestep)(ds_merged, t) for t in range(n_time)],
    scheduler='processes'
)
time_processes = time.time() - start
cape_processes = np.stack(results_processes, axis=-1)

# Correctness 
match_processes = np.allclose(result_seq.values, cape_processes, equal_nan=True)

# Summary table
print(f"{'Method':<30}{'Time (s)':<12}{'Speedup':<10}{'Matches seq?'}")
print(f"{'Sequential':<30}{time_seq:<12.2f}{'1.00x':<10}{'-'}")
print(f"{'Dask batched (processes)':<30}{time_processes:<12.2f}{time_seq/time_processes:<10.2f}{match_processes}")

Method                        Time (s)    Speedup   Matches seq?
Sequential                    200.25      1.00x     -
Dask batched (processes)      30.98       6.46      True


In [3]:
from src.features.meteorology import compute_cape_dask_batched
xrCape = compute_cape_dask_batched(ds_merged)

BRN


In [8]:
from src.features.meteorology import compute_brn

brn = compute_brn(xrCape, shear_features["bulk_shear_10m_500"])
brn_named = brn.rename("brn")

precip = ds_merged['tp']

all_features = xr.merge([shear_features, xrCape, brn_named, precip])
dfFeatures = all_features.to_dataframe()
dfFeatures = dfFeatures.drop(columns=['number', 'expver'])
dfFeatures = dfFeatures.reset_index()

print(dfFeatures['tp'].describe())
print((dfFeatures['tp'] == 0).sum())

print("Number of NaN Values:")
print(dfFeatures.isna().sum())

print(dfFeatures['tp'].quantile([0.90, 0.95, 0.99]))
print((dfFeatures['tp'] >= 0.001).sum())  # how many rows exceed 1mm

dfFeatures['heavy_rain'] = (dfFeatures['tp'] >= 0.001).astype(int)
print("===============================")
print(dfFeatures['heavy_rain'].value_counts())

count    2.881200e+04
mean     9.416592e-05
std      2.647649e-04
min      0.000000e+00
25%      4.768372e-07
50%      8.106232e-06
75%      4.959106e-05
max      3.624916e-03
Name: tp, dtype: float64
6615
Number of NaN Values:
valid_time            0
latitude              0
longitude             0
bulk_shear_10m_850    0
bulk_shear_850_500    0
bulk_shear_10m_500    0
dir_shear_10m_850     0
dir_shear_850_500     0
dir_shear_10m_500     0
cape                  0
brn                   0
tp                    0
dtype: int64
0.90    0.000242
0.95    0.000521
0.99    0.001401
Name: tp, dtype: float64
605
heavy_rain
0    28207
1      605
Name: count, dtype: int64


CHECKPOINT

In [9]:
#Checkpoint (so can start from here when restarting kernel)
dfFeatures.to_csv("../data/processed/era5_features_jan2020.csv", index=False)

In [10]:
from sklearn.model_selection import train_test_split

feature_columns = ["bulk_shear_10m_850", "bulk_shear_850_500", "bulk_shear_10m_500", "dir_shear_10m_850", "dir_shear_850_500", "dir_shear_10m_500", "cape", "brn"]
X = dfFeatures[feature_columns]
y = dfFeatures["heavy_rain"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(y_train.value_counts())
print(y_test.value_counts())

heavy_rain
0    22565
1      484
Name: count, dtype: int64
heavy_rain
0    5642
1     121
Name: count, dtype: int64


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

model = LogisticRegression(class_weight='balanced')
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(pd.Series(y_pred).value_counts())
print("Accuracy:", model.score(X_test, y_test))

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

0    5760
1       3
Name: count, dtype: int64
Accuracy: 0.9788304702411939
[[5640    2]
 [ 120    1]]
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      5642
           1       0.33      0.01      0.02       121

    accuracy                           0.98      5763
   macro avg       0.66      0.50      0.50      5763
weighted avg       0.97      0.98      0.97      5763

